В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням kNearestNeighboors, знайдемо оптимальні гіперпараметри для цього методу і зробимо базові ансамблі. Це дасть змогу порівняти перформанс моделі з попередніми вивченими методами.

0. Зчитайте дані `train.csv` та зробіть препроцесинг використовуючи написаний Вами скрипт `process_bank_churn.py` так, аби в результаті отримати дані в розбитті X_train, train_targets, X_val, val_targets для експериментів.

  Якщо Вам не вдалось реалізувати в завданні `2.3. Дерева прийняття рішень` скрипт `process_bank_churn.py` - можна скористатись готовим скриптом з запропонованого рішення того завдання.

In [1]:
import pandas as pd
from process_bank_churn import preprocess_data

raw_df = pd.read_csv('train.csv')

X_train, train_targets, X_val, val_targets, input_cols, scaler, encoder = preprocess_data(
    raw_df,
    scaler_numeric=True,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(train_targets.shape)
print(X_val.shape)
print(val_targets.shape)

(12000, 13)
(12000,)
(3000, 13)
(3000,)


1. Навчіть на цих даних класифікатор kNN з параметрами за замовченням і виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах. Зробіть заключення про отриману модель: вона хороша/погана, чи є high bias/high variance?

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

knn = KNeighborsClassifier()
knn.fit(X_train, train_targets)

train_probs = knn.predict_proba(X_train)[:, 1]
val_probs = knn.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(train_targets, train_probs)
val_auc = roc_auc_score(val_targets, val_probs)

print(f'Train AUROC: {train_auc:.4f}')
print(f'Validation AUROC: {val_auc:.4f}')
print(f'Difference: {train_auc - val_auc:.4f}')

Train AUROC: 0.9559
Validation AUROC: 0.8526
Difference: 0.1034


Висновок: модель KNN показує хорошу якість класифікації на валідаційній вибірці, проте значна різниця між Train AUROC та Validation AUROC свідчить про перенавчання (high variance). Ознак недонавчання (high bias) не спостерігається. Для покращення узагальнювальної здатності моделі варто здійснити підбір гіперпараметрів, зокрема кількості найближчих сусідів.

2. Використовуючи `GridSearchCV` знайдіть оптимальне значення параметра `n_neighbors` для класифікатора `kNN`. Псотавте крос валідацію на 5 фолдів.

  Після успішного завершення пошуку оптимального гіперпараметра
    - виведіть найкраще значення параметра
    - збережіть в окрему змінну `knn_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `knn_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пукнтом (2) цього завдання? Чи є вона краще за дерево прийняття рішень з попереднього ДЗ?

In [3]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors': range(1, 51)
}

grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, train_targets)

print('Best n_neighbors:', grid_search.best_params_['n_neighbors'])
print(f'Best CV AUROC: {grid_search.best_score_:.4f}')

knn_best = grid_search.best_estimator_

train_probs_best = knn_best.predict_proba(X_train)[:, 1]
val_probs_best = knn_best.predict_proba(X_val)[:, 1]

train_auc_best = roc_auc_score(train_targets, train_probs_best)
val_auc_best = roc_auc_score(val_targets, val_probs_best)

print(f'Train AUROC: {train_auc_best:.4f}')
print(f'Validation AUROC: {val_auc_best:.4f}')
print(f'Difference: {train_auc_best - val_auc_best:.4f}')

Best n_neighbors: 19
Best CV AUROC: 0.8853
Train AUROC: 0.9227
Validation AUROC: 0.8908
Difference: 0.0319


Висновок: за допомогою GridSearchCV визначено оптимальне значення n_neighbors=19. Після підбору гіперпараметра Validation AUROC зріс з 0.8526 до 0.8908, а різниця між тренувальною та валідаційною вибірками зменшилася з 0.1034 до 0.0319. Це свідчить про зменшення перенавчання (high variance) та покращення узагальнювальної здатності моделі. Проте дерево прийняття рішень показало кращу якість класифікації.

3. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `GridSearchCV` за сіткою параметрів
  - `max_depth` від 1 до 20 з кроком 2
  - `max_leaf_nodes` від 2 до 10 з кроком 1

  Обовʼязково при цьому ініціюйте модель з фіксацією `random_state`.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `dt_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли вручну?

In [4]:
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

param_grid_dt = {
    'max_depth': range(1, 21, 2),
    'max_leaf_nodes': range(2, 11)
}

grid_search_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid_dt,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

start_time = time.time()

grid_search_dt.fit(X_train, train_targets)

end_time = time.time()

print(f'Execution time: {end_time - start_time:.2f} seconds')

print('Best parameters:', grid_search_dt.best_params_)
print(f'Best CV AUROC: {grid_search_dt.best_score_:.4f}')

dt_best = grid_search_dt.best_estimator_

train_probs_dt = dt_best.predict_proba(X_train)[:, 1]
val_probs_dt = dt_best.predict_proba(X_val)[:, 1]

train_auc_dt = roc_auc_score(train_targets, train_probs_dt)
val_auc_dt = roc_auc_score(val_targets, val_probs_dt)

print(f'Train AUROC: {train_auc_dt:.4f}')
print(f'Validation AUROC: {val_auc_dt:.4f}')
print(f'Difference: {train_auc_dt - val_auc_dt:.4f}')

Execution time: 16.10 seconds
Best parameters: {'max_depth': 5, 'max_leaf_nodes': 10}
Best CV AUROC: 0.9014
Train AUROC: 0.9015
Validation AUROC: 0.9002
Difference: 0.0013


Висновок: за допомогою GridSearchCV знайдено оптимальні гіперпараметри max_depth=5 та max_leaf_nodes=10. Модель показує хорошу якість класифікації, а мінімальна різниця між Train AUROC (0.9015) та Validation AUROC (0.9002) свідчить про відсутність явного перенавчання. Проте модель із ручним підбором гіперпараметрів показала кращий Validation AUROC (0.9246). Можливою причиною є обмежений діапазон гіперпараметрів у GridSearchCV.

4. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `RandomizedSearchCV` за заданою сіткою параметрів і кількість ітерацій 40.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, зафіксуйте `random_seed` процедури крос валідації та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_random_search_best` найкращу модель, знайдену з `RandomizedSearchCV`
    - оцініть якість передбачень  `dt_random_search_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли з `GridSearch`?
    - проаналізуйте параметри `dt_random_search_best` і порівняйте з параметрами `dt_best` - яку бачите відмінність? Ця вправа потрібна аби зрозуміти, як різні налаштування `DecisionTreeClassifier` впливають на якість моделі.

In [5]:
import numpy as np
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score

params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': np.arange(1, 20),
    'max_leaf_nodes': np.arange(2, 20),
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

random_search_dt = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=params_dt,
    n_iter=40,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

random_search_dt.fit(X_train, train_targets)

end_time = time.time()

print(f'Execution time: {end_time - start_time:.2f} seconds')

print('Best parameters:', random_search_dt.best_params_)
print(f'Best CV AUROC: {random_search_dt.best_score_:.4f}')

dt_random_search_best = random_search_dt.best_estimator_

train_probs_random = dt_random_search_best.predict_proba(X_train)[:, 1]
val_probs_random = dt_random_search_best.predict_proba(X_val)[:, 1]

train_auc_random = roc_auc_score(train_targets, train_probs_random)
val_auc_random = roc_auc_score(val_targets, val_probs_random)

print(f'Train AUROC: {train_auc_random:.4f}')
print(f'Validation AUROC: {val_auc_random:.4f}')
print(f'Difference: {train_auc_random - val_auc_random:.4f}')

comparison = pd.DataFrame({
    'GridSearchCV': dt_best.get_params(),
    'RandomizedSearchCV': dt_random_search_best.get_params()
})

display(comparison.loc[list(params_dt.keys())])

Execution time: 11.73 seconds
Best parameters: {'splitter': 'best', 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_leaf_nodes': np.int64(14), 'max_features': None, 'max_depth': np.int64(16), 'criterion': 'entropy'}
Best CV AUROC: 0.9103
Train AUROC: 0.9169
Validation AUROC: 0.9166
Difference: 0.0003


,GridSearchCV,RandomizedSearchCV
criterion,gini,entropy
splitter,best,best
max_depth,5,16
max_leaf_nodes,10,14
min_samples_split,2,20
min_samples_leaf,1,2
max_features,None,None


Висновок: модель RandomizedSearchCV показала кращий Validation AUROC (0.9166 проти 0.9002) та менший час пошуку (11.73 проти 16.10 с). Збільшення глибини дерева та кількості листків дозволяє виявляти складніші закономірності, а збільшення min_samples_split і min_samples_leaf допомагає контролювати перенавчання. Також критерій gini змінився на entropy. Ознак суттєвого перенавчання не спостерігається.

5. Якщо у Вас вийшла метрика `AUROC` в цій серії експериментів - зробіть ще один `submission` на Kaggle і додайте код для цього і скріншот скора на публічному лідерборді нижче.

  Сподіваюсь на цьому етапі ви вже відчули себе справжнім дослідником 😉